# Visualize ERA5 Grid → Polygon Mapping (Basins & Watersheds) + Coverage QA

This notebook **uses only the mapping artifact**  
`grid_id, latitude, longitude, basin_id, watershed_id, outside_flag`  
to visualize and QA the spatial assignment of ERA5 grid cells.

### What this notebook does
- **Overlay plots** (static):
  - Points colored by **basin** and by **watershed** (outside points omitted; IDs treated as categories).
  - Saved as separate PNGs and also displayed side-by-side in the notebook.
- **Coverage QA**:
  - Counts how many ERA5 grid cells fall inside **each polygon**.
  - Flags **low-coverage** polygons (≤ threshold cells; default 1).
  - Produces a **choropleth map** of counts and **CSV reports**.

### Inputs
- Mapping file:  
  `PROJECT_ROOT / data/spatial/grid_mapping/era5_grid_to_polygons.parquet`
- Polygons:  
  `PROJECT_ROOT / data/boundaries/basins/`  
  `PROJECT_ROOT / data/boundaries/186_watershed/`

### Outputs (written relative to the repo root)
**Visualization**
- `data/spatial/grid_mapping/viz/mapping_overlay_basin.png`
- `data/spatial/grid_mapping/viz/mapping_overlay_watershed.png`

**Coverage QA — Basins**
- `data/spatial/grid_mapping/qa/basin_coverage_counts.csv`  
  (rows: `polygon_id, n_cells`)
- `data/spatial/grid_mapping/qa/basin_low_coverage_le_1.csv`  
  (polygons with `n_cells ≤ 1`; threshold is configurable)
- `data/spatial/grid_mapping/qa/basin_coverage_choropleth.png`

**Coverage QA — Watersheds**
- `data/spatial/grid_mapping/qa/watershed_coverage_counts.csv`
- `data/spatial/grid_mapping/qa/watershed_low_coverage_le_1.csv`
- `data/spatial/grid_mapping/qa/watershed_coverage_choropleth.png`

> Notes:
> - Legends are shown for basins (few categories) and hidden for watersheds (many categories).
> - All paths are constructed from `PROJECT_ROOT` so the notebook works no matter where it’s run.


In [1]:
# !pip install pandas geopandas shapely pyproj fiona matplotlib


In [2]:
from pathlib import Path
import os, subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "data").exists() and (p / "code").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


In [3]:
# Mapping artifact (Stage A output)
MAPPING_PARQUET = PROJECT_ROOT / "data/spatial/grid_mapping/era5_grid_to_polygons.parquet"

# Shapefile folders
BASINS_DIR      = PROJECT_ROOT / "data/boundaries/basins"
WATERSHEDS_DIR  = PROJECT_ROOT / "data/boundaries/186_watershed"

print("Mapping file:", MAPPING_PARQUET)
print("Basins dir:", BASINS_DIR)
print("Watersheds dir:", WATERSHEDS_DIR)


Mapping file: /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/era5_grid_to_polygons.parquet
Basins dir: /Users/liuq13/bhutan_climate_modeling/data/boundaries/basins
Watersheds dir: /Users/liuq13/bhutan_climate_modeling/data/boundaries/186_watershed


In [4]:
import pandas as pd, geopandas as gpd

mp = pd.read_parquet(MAPPING_PARQUET, columns=[
    "grid_id","latitude","longitude","basin_id","watershed_id","outside_flag"
])
print("Mapping rows:", len(mp), "| Unique grid cells:", mp["grid_id"].nunique())

# Points from mapping (EPSG:4326)
pts = gpd.GeoDataFrame(
    mp, geometry=gpd.points_from_xy(mp.longitude, mp.latitude), crs="EPSG:4326"
)

# Quick sanity: how many points are outside any polygon?
print("Outside (no basin & no watershed):", int(pts["outside_flag"].sum()))


Mapping rows: 135 | Unique grid cells: 135
Outside (no basin & no watershed): 61


In [5]:
import glob

def _pick(vecdir: Path):
    c = glob.glob(str(vecdir / "*.shp")) + glob.glob(str(vecdir / "*.gpkg"))
    if not c:
        raise FileNotFoundError(f"No vector file found in {vecdir}")
    return c[0]

basins_path = _pick(BASINS_DIR)
watersheds_path = _pick(WATERSHEDS_DIR)

basins = gpd.read_file(basins_path).to_crs(4326)
watersheds = gpd.read_file(watersheds_path).to_crs(4326)

print("Basins file:", basins_path, "| Features:", len(basins))
print("Watersheds file:", watersheds_path, "| Features:", len(watersheds))


Basins file: /Users/liuq13/bhutan_climate_modeling/data/boundaries/basins/Basin boundary.shp | Features: 10
Watersheds file: /Users/liuq13/bhutan_climate_modeling/data/boundaries/186_watershed/186 Watershed boundary.shp | Features: 186


In [6]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_overlay_to_png(points_gdf, polygons_gdf, id_col: str, title: str, out_png,
                        show_legend=True, markersize=22, omit_outside=True):
    g = points_gdf.copy()

    # ---- omit points outside polygons ----
    if omit_outside:
        if "outside_flag" in g.columns:
            g = g.loc[~g["outside_flag"]].copy()
        else:
            g = g.loc[g[id_col].notna()].copy()

    # ---- treat IDs as categories (discrete colors; no continuous colorbar) ----
    # cast to strings (robust for int/float IDs)
    g["_cat"] = g[id_col].astype("Int64").astype("string")
    g["_cat"] = pd.Categorical(g["_cat"])
    ncat = g["_cat"].nunique()
    show_legend = show_legend and (ncat <= 30)

    fig, ax = plt.subplots(figsize=(7, 7))
    polygons_gdf.boundary.plot(ax=ax, linewidth=0.6)
    g.plot(ax=ax, column="_cat", legend=show_legend,
           legend_kwds={"bbox_to_anchor": (1.02, 1), "loc": "upper left", "title": id_col},
           markersize=markersize, alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    return out_png


In [7]:
# Define output paths and render PNGs
OUT_DIR = PROJECT_ROOT / "data" / "spatial" / "grid_mapping" / "viz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

basin_png = OUT_DIR / "mapping_overlay_basin.png"
ws_png    = OUT_DIR / "mapping_overlay_watershed.png"

# Basins: few categories → legend ON
plot_overlay_to_png(pts, basins, "basin_id",
    "ERA5 grid cells assigned to Basins (from mapping file)",
    basin_png, show_legend=True)

# Watersheds: many categories → legend OFF (to avoid a wall of labels)
plot_overlay_to_png(pts, watersheds, "watershed_id",
    "ERA5 grid cells assigned to Watersheds (from mapping file)",
    ws_png, show_legend=False, markersize=18)

basin_png, ws_png

(PosixPath('/Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/viz/mapping_overlay_basin.png'),
 PosixPath('/Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/viz/mapping_overlay_watershed.png'))

- These plots reflect exactly what’s in the **mapping file** (no raw ERA5 data used).
- If you see points on boundaries that look mis-assigned, consider upgrading Stage A to a **fractional (area-weighted) mapping**.
- You can switch to interactive maps with Folium if you want pan/zoom/popups.


In [8]:
# Coverage QA helpers
import glob
import pandas as pd, geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

CANDIDATE_ID_COLS = [
    "polygon_id","basin_id","watershed_id","WS_ID","BASIN_ID","Basin_ID",
    "OBJECTID","FID","ID","id","CODE","code","NAME","Name","name"
]

def _pick_vector(vecdir: Path):
    paths = glob.glob(str(vecdir / "*.shp")) + glob.glob(str(vecdir / "*.gpkg"))
    if not paths:
        raise FileNotFoundError(f"No .shp/.gpkg found in {vecdir}")
    return paths[0]

def read_polygons_norm(dir_path: Path) -> gpd.GeoDataFrame:
    """Load polygons and expose a unified 'polygon_id' column for joining."""
    vec = _pick_vector(dir_path)
    gdf = gpd.read_file(vec)
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    # choose an id col or create one
    id_col = next((c for c in CANDIDATE_ID_COLS if c in gdf.columns), None)
    if id_col is None:
        gdf = gdf.reset_index().rename(columns={"index":"polygon_id"})
        id_col = "polygon_id"
    return gdf.rename(columns={id_col:"polygon_id"})[["polygon_id","geometry"]]


In [9]:
def coverage_qa(mp: pd.DataFrame, poly_dir: Path, id_in_mapping: str,
                layer_name: str, threshold: int = 1):
    """
    mp              : mapping dataframe with columns [<id_in_mapping>, outside_flag, latitude, longitude]
    poly_dir        : folder containing the shapefile/gpkg for this layer
    id_in_mapping   : 'basin_id' or 'watershed_id'
    layer_name      : label for outputs, e.g., 'basin' or 'watershed'
    threshold       : flag polygons with n_cells <= threshold
    """
    # 1) counts
    mp_use = mp.loc[(~mp["outside_flag"]) & (mp[id_in_mapping].notna())].copy()
    counts = (mp_use.groupby(id_in_mapping)
                    .size().rename("n_cells")
                    .reset_index())

    # 2) load polygons with unified 'polygon_id'
    polys = read_polygons_norm(poly_dir)

    # 3) join counts -> polygons
    joined = polys.merge(counts, left_on="polygon_id", right_on=id_in_mapping, how="left")
    joined["n_cells"] = joined["n_cells"].fillna(0).astype(int)

    # 4) summary + flags
    total = len(joined)
    zero  = int((joined["n_cells"] == 0).sum())
    low   = int((joined["n_cells"] <= threshold).sum())
    print(f"[{layer_name}] polygons: {total} | zero-cell: {zero} | ≤{threshold} cells: {low}")

    # 5) save CSVs
    qa_dir = PROJECT_ROOT / "data" / "spatial" / "grid_mapping" / "qa"
    qa_dir.mkdir(parents=True, exist_ok=True)
    out_all = qa_dir / f"{layer_name}_coverage_counts.csv"
    out_low = qa_dir / f"{layer_name}_low_coverage_le_{threshold}.csv"
    joined[["polygon_id","n_cells"]].to_csv(out_all, index=False)
    joined.loc[joined["n_cells"] <= threshold, ["polygon_id","n_cells"]].to_csv(out_low, index=False)
    print("Saved:", out_all, "|", out_low)

    # 6) choropleth (static)
    fig, ax = plt.subplots(figsize=(8,6))
    joined.plot(column="n_cells", legend=True, edgecolor="grey", linewidth=0.4, ax=ax)
    ax.set_title(f"#{layer_name.title()} grid cells per {layer_name} (from mapping)")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    fig.tight_layout()

    # save PNG
    png_path = qa_dir / f"{layer_name}_coverage_choropleth.png"
    fig.savefig(png_path, dpi=150)
    plt.close(fig)
    print("Saved:", png_path)

    return joined[["polygon_id","n_cells"]].sort_values("n_cells")


In [10]:
# Ensure mapping file is loaded in this notebook
try:
    mp
except NameError:
    import pandas as pd
    MAPPING_PARQUET = PROJECT_ROOT / "data/spatial/grid_mapping/era5_grid_to_polygons.parquet"
    mp = pd.read_parquet(MAPPING_PARQUET)

# Paths to shapefile folders (reuse from earlier cells if defined)
BASINS_DIR      = PROJECT_ROOT / "data/boundaries/basins"
WATERSHEDS_DIR  = PROJECT_ROOT / "data/boundaries/186_watershed"

# Run coverage QA
basin_counts = coverage_qa(mp, BASINS_DIR,     id_in_mapping="basin_id",     layer_name="basin",     threshold=1)
ws_counts    = coverage_qa(mp, WATERSHEDS_DIR, id_in_mapping="watershed_id", layer_name="watershed", threshold=1)

# Peek at the lowest-coverage polygons
display(basin_counts.head(10))
display(ws_counts.head(15))


[basin] polygons: 10 | zero-cell: 2 | ≤1 cells: 3
Saved: /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/basin_coverage_counts.csv | /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/basin_low_coverage_le_1.csv
Saved: /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/basin_coverage_choropleth.png
[watershed] polygons: 186 | zero-cell: 132 | ≤1 cells: 184
Saved: /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/watershed_coverage_counts.csv | /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/watershed_low_coverage_le_1.csv
Saved: /Users/liuq13/bhutan_climate_modeling/data/spatial/grid_mapping/qa/watershed_coverage_choropleth.png


,polygon_id,n_cells
0,0,0
2,2,0
4,4,1
1,1,2
9,9,3
5,5,5
6,6,8
3,3,11
8,8,14
7,7,28


,polygon_id,n_cells
0,0,0
109,109,0
111,111,0
113,113,0
114,114,0
116,116,0
117,117,0
118,118,0
119,119,0
120,120,0
